# Decision Tree
Decision Tree implementation using CART algorithm

In [8]:
%pip install scikit-learn
%pip install biopython

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.model_selection import GroupShuffleSplit
from Bio.Align import substitution_matrices
from collections import Counter
from treenode import TreeNode
from sklearn.model_selection import train_test_split

# Load and preprocess data

In [7]:
df = pd.read_csv("../data/features/processed_data_with_position_specific_features.csv")

#Drop non feature columns
DROP_COLS = ["peptide", "hla_sequence", "index"]
TARGET_COL = "Label"

df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

#Identify categorical columns
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
cat_cols = [c for c in cat_cols if c != TARGET_COL]
print(f"Encoding categorical columns: {cat_cols}")

df = pd.get_dummies(df, columns=cat_cols) 

# Split x and y values
y = df[TARGET_COL].values
X = df.drop(columns=[TARGET_COL])

print(df.columns.tolist()) #check what columns exist

print(f"Features: {X.shape}, Classes: {dict(zip(*np.unique(y, return_counts=True)))}")



train_idx = np.load("../data/splits/train_idx.npy")
val_idx = np.load("../data/splits/val_idx.npy")
test_idx = np.load("../data/splits/test_idx.npy")

X_train, y_train = X.iloc[train_idx], y[train_idx]
X_val, y_val     = X.iloc[val_idx], y[val_idx]
X_test, y_test   = X.iloc[test_idx], y[test_idx]

# Save feature names and convert to NumPy
feature_names = X.columns.tolist()
X = X.values

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Encoding categorical columns: ['HLA']
['Label', 'PeptidePos_p1_f1', 'PeptidePos_p1_f2', 'PeptidePos_p1_f3', 'PeptidePos_p1_f4', 'PeptidePos_p1_f5', 'PeptidePos_p1_f6', 'PeptidePos_p1_f7', 'PeptidePos_p1_f8', 'PeptidePos_p1_f9', 'PeptidePos_p1_f10', 'PeptidePos_p1_f11', 'PeptidePos_p1_f12', 'PeptidePos_p1_f13', 'PeptidePos_p1_f14', 'PeptidePos_p1_f15', 'PeptidePos_p1_f16', 'PeptidePos_p1_f17', 'PeptidePos_p1_f18', 'PeptidePos_p2_f1', 'PeptidePos_p2_f2', 'PeptidePos_p2_f3', 'PeptidePos_p2_f4', 'PeptidePos_p2_f5', 'PeptidePos_p2_f6', 'PeptidePos_p2_f7', 'PeptidePos_p2_f8', 'PeptidePos_p2_f9', 'PeptidePos_p2_f10', 'PeptidePos_p2_f11', 'PeptidePos_p2_f12', 'PeptidePos_p2_f13', 'PeptidePos_p2_f14', 'PeptidePos_p2_f15', 'PeptidePos_p2_f16', 'PeptidePos_p2_f17', 'PeptidePos_p2_f18', 'PeptidePos_p3_f1', 'PeptidePos_p3_f2', 'PeptidePos_p3_f3', 'PeptidePos_p3_f4', 'PeptidePos_p3_f5', 'PeptidePos_p3_f6', 'PeptidePos_p3_f7', 'PeptidePos_p3_f8', 'PeptidePos_p3_f9', 'PeptidePos_p3_f10', 'PeptidePos_p

/var/folders/zp/5ww68fzx2x5dvm0pzf3hdprm0000gn/T/ipykernel_2094/129062166.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()


# Decision Tree 

In [8]:
#Decision Tree implementation is based on the CART Algorithm
#Compute Gini impurity, which measures how mixed the labels are (0 = pure, higher = more mixed).
def gini(y):
    if len(y) == 0:
        return 0.0
    counts = np.bincount(y)
    probs  = counts / len(y)
    return 1.0 - np.sum(probs ** 2)

#Compute Gini-based information gain, which measures how much a split improves purity. 
def information_gain(y_parent, y_left, y_right):
    n = len(y_parent)
    weighted_child = (len(y_left) / n)  * gini(y_left) + \
                     (len(y_right) / n) * gini(y_right)
    return gini(y_parent) - weighted_child

# Find the best feature and threshold to split on to maximize information gain 
def best_split(X, y):
    best_gain = -1
    best_feat, best_thresh = None, None

    for feat in range(X.shape[1]):
        thresholds = np.unique(X[:, feat])

        for thresh in thresholds:
            left_mask  = X[:, feat] <= thresh
            right_mask = ~left_mask

            if left_mask.sum() == 0 or right_mask.sum() == 0:
                continue

            gain = information_gain(y, y[left_mask], y[right_mask])

            if gain > best_gain:
                best_gain   = gain
                best_feat   = feat
                best_thresh = thresh

    return best_feat, best_thresh, best_gain

#Create a leaf node containing Create a leaf node containing class counts and class probabilities
def make_leaf(y, n_samples):
    unique, counts = np.unique(y, return_counts=True)
    label_counts = dict(zip(unique.tolist(), counts.tolist()))
    probs = {k: v / n_samples for k, v in label_counts.items()}

    return TreeNode(
        n_samples=n_samples,
        prediction_probs=probs,
        label_counts=label_counts
    )


#Recursively build the decision tree using CART algorithm. Stops when: max depth is reached, node is pure, or not enough samples
def build_tree(X, y, depth=0, max_depth=10, min_samples_split=2, min_samples_leaf=1):
    n_samples = len(y)

    #Check stopping conditions
    if (
        depth >= max_depth
        or n_samples < min_samples_split
        or len(set(y)) == 1
    ):
        return make_leaf(y, n_samples)

    #Find best split
    feat, thresh, gain = best_split(X, y)

    #Stop if no useful split found
    if feat is None or gain <= 0:
        return make_leaf(y, n_samples)

    #Apply split
    left_mask = X[:, feat] <= thresh
    right_mask = ~left_mask

    #Enforce minimum leaf size
    if left_mask.sum() < min_samples_leaf or right_mask.sum() < min_samples_leaf:
        return make_leaf(y, n_samples)

    #Create decision node
    node = TreeNode(
        feature_idx=feat,
        feature_val=thresh,
        information_gain=gain,
        n_samples=n_samples
    )

    #Recursively build left subtree 
    node.left = build_tree(
        X[left_mask], y[left_mask],
        depth + 1, max_depth, min_samples_split, min_samples_leaf
    )

    #Recursively build right subtree
    node.right = build_tree(
        X[right_mask], y[right_mask],
        depth + 1, max_depth, min_samples_split, min_samples_leaf
    )

    return node

#Predict label for a single sample by traversing the tree
def predict_one(node, x):
    if node.is_leaf:
        return max(node.prediction_probs, key=node.prediction_probs.get)

    if x[node.feature_idx] <= node.feature_val:
        return predict_one(node.left, x)
    else:
        return predict_one(node.right, x)

#Predict labels for multiple samples
def predict(root, X):
    return np.array([predict_one(root, x) for x in X])


#Aggregate feature importance based on information gain
def get_feature_importances(node, n_features):

    importances = np.zeros(n_features)

    def walk(n):
        if n is None or n.is_leaf:
            return

        importances[n.feature_idx] += n.feature_importance

        walk(n.left)
        walk(n.right)

    walk(node)

    total = importances.sum()
    return importances / total if total > 0 else importances

#CART decision tree classifier implementation
class CARTClassifier:
    #Store hyperparameters
    def __init__(self, max_depth=10, min_samples_split=2, min_samples_leaf=1):
        self.max_depth         = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf  = min_samples_leaf
        self.root              = None
        self.feature_names_    = None

    #Train the decision tree
    def fit(self, X, y, feature_names=None):
        self.root           = build_tree(X, y,
                                         max_depth=self.max_depth,
                                         min_samples_split=self.min_samples_split,
                                         min_samples_leaf=self.min_samples_leaf)
        self.feature_names_ = feature_names
        return self

    #Predict labels
    def predict(self, X):
        return predict(self.root, X)

    #Estimate accuracy (preliminary)
    def score(self, X, y):
        return np.mean(self.predict(X) == y)

    #Return sorted feature importance values
    def feature_importances(self):
        imps = get_feature_importances(self.root, len(self.feature_names_ or []))
        if self.feature_names_:
            return pd.Series(imps, index=self.feature_names_).sort_values(ascending=False)
        return imps
    

#Training and evaluating the model 

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cart = CARTClassifier(max_depth=6, min_samples_split=5, min_samples_leaf=2)
cart.fit(X_train, y_train, feature_names=feature_names)

print(f"Train accuracy: {cart.score(X_train, y_train):.4f}")
print(f"Test  accuracy: {cart.score(X_test,  y_test):.4f}")

print("\nTop 10 most important features:")
print(cart.feature_importances().head(10))

Train accuracy: 0.7891
Test  accuracy: 0.7620

Top 10 most important features:
HLA_F3               0.547881
HLA_VSTPV2           0.119259
HLA_BLOSUM6          0.099633
HLA_PD1              0.044983
PeptidePos_p4_f17    0.031409
HLA_F4               0.021565
HLA_BLOSUM3          0.013586
PeptidePos_p10_f1    0.009725
PeptidePos_p2_f5     0.009514
PeptidePos_p10_f4    0.009471
dtype: float64
